In [1]:
# Install required libraries silently
!pip install -q -r requirements.txt

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer
from transformers import TrainingArguments

# Ensure we are using the GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavio

In [2]:
# Create a reusable, recursive parser for PyTorch nn.Modules
def build_model_tree(module, name="Root"):
    """Recursively inspects the model and builds a hierarchical dictionary."""
    tree = {
        "name": name,
        "type": module.__class__.__name__,
        "params": sum(p.numel() for p in module.parameters()),
        "trainable_params": sum(p.numel() for p in module.parameters() if p.requires_grad),
        "children": []
    }
    for child_name, child_module in module.named_children():
        tree["children"].append(build_model_tree(child_module, child_name))
    return tree

def print_model_tree(tree, indent=0, max_depth=3, current_depth=0):
    """Visually prints the hierarchical dictionary."""
    if current_depth > max_depth:
        return

    pad = "  " * indent
    bullet = "├──" if indent > 0 else "■"

    print(f"{pad}{bullet} {tree['name']} ({tree['type']}) | Params: {tree['params']:,}")

    for child in tree["children"]:
        print_model_tree(child, indent + 1, max_depth, current_depth + 1)

# Load base models (TinyLlama and GPT-2)
print("Loading TinyLlama...")
llama_model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.float16, device_map="auto")

print("\n--- TinyLlama Architecture Tree ---")
llama_tree = build_model_tree(llama_model, "TinyLlama-1.1B")
print_model_tree(llama_tree, max_depth=3)

Loading TinyLlama...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


--- TinyLlama Architecture Tree ---
■ TinyLlama-1.1B (LlamaForCausalLM) | Params: 1,100,048,384
  ├── model (LlamaModel) | Params: 1,034,512,384
    ├── embed_tokens (Embedding) | Params: 65,536,000
    ├── layers (ModuleList) | Params: 968,974,336
      ├── 0 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 1 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 2 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 3 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 4 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 5 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 6 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 7 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 8 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 9 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 10 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 11 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 12 (LlamaDecoderLayer) | Params: 44,044,288
      ├── 13 (LlamaDecoderLayer) | Params: 44,044,288
    

In [3]:
# We will use 4-bit quantization to fit the fine-tuning on a Colab T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Reload TinyLlama with 4-bit config and load its tokenizer
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto"
)

# Load a small slice of IMDB for a quick Sentiment Analysis fine-tune
dataset = load_dataset("imdb", split="train[:500]")

# Format the prompt for causal LM
def format_prompt(example):
    sentiment = "Positive" if example["label"] == 1 else "Negative"
    text = f"<|user|>\nAnalyze the sentiment of this review: {example['text'][:500]}<|end|>\n<|assistant|>\nThe sentiment is {sentiment}.<|end|>"
    return {"text": text}

formatted_dataset = dataset.map(format_prompt)

# Setup LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], # Targeting attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
print("\nTrainable parameters after applying LoRA:")
model.print_trainable_parameters()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Trainable parameters after applying LoRA:
trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.20437245579516677


In [5]:
# Keep training incredibly short (1 epoch) for the sake of the assignment
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    max_grad_norm=0.3,
    max_steps=50,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=256,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()

# Save the adapter
new_model_adapter = "tinyllama-imdb-lora"
trainer.model.save_pretrained(new_model_adapter)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Starting training...


Step,Training Loss
10,2.791100
20,2.311500
30,2.174600
40,2.182500
50,2.094700


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
# Safely free up VRAM, ignoring the command if they were already deleted
try:
    del model
    del trainer
except NameError:
    pass

torch.cuda.empty_cache()

print("Reloading base model for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Define the test prompt
prompt = "<|user|>\nAnalyze the sentiment of this review: The acting was terrible and the plot made no sense.<|end|>\n<|assistant|>\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# 1. Test the BASE model behavior BEFORE merging
print("\n--- Output from BASE Model (Before LoRA) ---")
base_outputs = base_model.generate(**inputs, max_new_tokens=20, temperature=0.1, do_sample=True)
print(tokenizer.decode(base_outputs[0], skip_special_tokens=True))

# 2. Merge adapter with base model
print("\nMerging adapter weights...")
peft_model = PeftModel.from_pretrained(base_model, new_model_adapter)
merged_model = peft_model.merge_and_unload()

# 3. Test the MERGED model behavior AFTER merging
print("\n--- Output from MERGED Model (After LoRA) ---")
merged_outputs = merged_model.generate(**inputs, max_new_tokens=20, temperature=0.1, do_sample=True)
print(tokenizer.decode(merged_outputs[0], skip_special_tokens=True))

Reloading base model for merging...

--- Output from BASE Model (Before LoRA) ---
<|user|>
Analyze the sentiment of this review: The acting was terrible and the plot made no sense.<|end|>
<|assistant|>
The reviewer's sentiment about the acting and plot is negative. The reviewer describes the acting

Merging adapter weights...

--- Output from MERGED Model (After LoRA) ---
<|user|>
Analyze the sentiment of this review: The acting was terrible and the plot made no sense.<|end|>
<|assistant|>
The sentiment is Negative.<|end|>
<|end|>
<|user
